In [41]:
import requests
import json
import pandas as pd
import googlemaps
from datetime import datetime
import numpy as np
from math import radians, cos, sin, asin, sqrt



In [46]:
house = pd.DataFrame({
    'id': [1],
    'latitude': [40.1930597],
    'longitude': [44.5022556]
})

df = pd.DataFrame({
    'id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'station_name': ['Barekamutyun', 'Baghramyan', 'Yeritasardakan', 'Republic Square', 'Zoravar Andranik', 'Sasuntsi Davit', 'Gortsaranain','Shengavit', 'Charbakh', 'Garegin Nzhdeh Square'],
    'latitude': [40.197446,40.1918167,40.1873081, 40.1785331, 40.1699272, 40.1552243, 40.1410611, 40.145224, 40.1405941, 40.1504761],
    'longitude': [44.493467,44.5038091, 44.5191743, 44.5134433, 44.5112603, 44.5058011, 44.4964173, 44.48483, 44.4685333, 44.4816183]
})


## Calculate using OpenRouteService

In [47]:
for house_idx, house_row in house.iterrows():
    # Set the start coordinates to the current house coordinates
    start_lon = house_row['longitude']
    start_lat = house_row['latitude']
    
    # Initialize the closest distance and closest station ID to large values
    closest_distance = float('inf')
    closest_station_id = -1
    
    # Loop through each row in the df table
    for df_idx, df_row in df.iterrows():
        # Set the end coordinates to the current metro station coordinates
        end_lon = df_row['longitude']
        end_lat = df_row['latitude']
        
        # Define the API URL and parameters
        url = 'https://api.openrouteservice.org/v2/directions/foot-walking'
        params = {
            'api_key': '***REMOVED-ORS-API-KEY***',
            'start': f'{start_lon},{start_lat}',
            'end': f'{end_lon},{end_lat}'
        }

        # Make the API request
        response = requests.get(url, params=params)

        # Parse the response JSON
        data = json.loads(response.text)

        # Extract the distance from the response
        distance = data['features'][0]['properties']['segments'][0]['distance']
        print(distance)
        # Update the closest distance and closest station ID if the current distance is smaller
        if distance < closest_distance:
            closest_distance = distance
            closest_station_id = df_row['id']
    
    # Store the closest station ID in the new column for the current house row
    house.loc[house_idx, 'closest_station_id'] = closest_station_id

# Save the updated house table to a new CSV file
house.to_csv('house_with_closest_station.csv', index=False)

1049.4
555.3
1718.4
2284.2
3346.7
5202.6
6604.9
7063.8
9601.3
6934.2


## Calculate using Google Routing API

In [37]:
import googlemaps
from datetime import datetime
import pandas as pd

# Function to calculate walking distance between two points using the Google Maps API
def get_walking_distance(start_lon, start_lat, end_lon, end_lat):
    # Set up the Google Maps API client
    gmaps = googlemaps.Client(key='***REMOVED-GOOGLE-API-KEY***')

    # Define the start and end coordinates
    start = (start_lat, start_lon)
    end = (end_lat, end_lon)

    # Make the API request
    now = datetime.now()
    directions = gmaps.directions(start, end, mode='walking', departure_time=now)

    # Extract the distance from the response
    distance = directions[0]['legs'][0]['distance']['value']

    # Return the distance in meters
    return distance


# Sample house and metro station coordinates
houses = pd.DataFrame({
    'id': [1],
    'lat': [40.1930597],
    'lon': [44.5022556]
})

stations = pd.DataFrame({
    'id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'station_name': ['Barekamutyun', 'Baghramyan', 'Yeritasardakan', 'Republic Square', 'Zoravar Andranik', 'Sasuntsi Davit', 'Gortsaranain','Shengavit', 'Charbakh', 'Garegin Nzhdeh Square'],
    'lat': [40.197446,40.1918167,40.1873081, 40.1785331, 40.1699272, 40.1552243, 40.1410611, 40.145224, 40.1405941, 40.1504761],
    'lon': [44.493467,44.5038091, 44.5191743, 44.5134433, 44.5112603, 44.5058011, 44.4964173, 44.48483, 44.4685333, 44.4816183]
})

In [40]:

# Calculate the walking distance from each house to each metro station
for i, house in houses.iterrows():
    distances = []
    for j, station in stations.iterrows():
        distance = get_walking_distance(house['lon'], house['lat'], station['lon'], station['lat'])
        distances.append(distance)
    stations['distance'] = distances
    print(stations['distance'])
    closest_station = stations.loc[stations['distance'].idxmin(), 'station_name']
    closest_dist = stations.loc[stations['distance'].idxmin(), 'distance']
    houses.at[i, 'closest_station'] = closest_station
    
    houses.at[i, 'closest_distance'] = closest_dist

# Print the houses table with the closest metro station
print(houses)


0    1056
1     329
2    1703
3    2287
4    3377
5    5039
6    6747
7    7184
8    9611
9    6960
Name: distance, dtype: int64
   id       lat        lon closest_station  closest_distance
0   1  40.19306  44.502256      Baghramyan             329.0


In [43]:
dist_matrix = np.zeros((len(houses), len(stations)))
for i in range(len(houses)):
    house = houses.iloc[i]
    distances = []
    for j in range(len(stations)):
        station = stations.iloc[j]
        distance = get_walking_distance(house['lon'], house['lat'], station['lon'], station['lat'])
        distances.append(distance)
    dist_matrix[i] = distances

# Find the index of the closest metro station for each house
closest_station_indices = np.argmin(dist_matrix, axis=1)

# Add the closest metro station to the houses table
houses['closest_station'] = stations.iloc[closest_station_indices]['id'].values

# Print the houses table with the closest metro station
print(houses)

   id       lat        lon  closest_station  closest_distance
0   1  40.19306  44.502256                2             329.0


## Calculating distance based on haversine formula

In [74]:
# Load the houses and stations tables into pandas dataframes
houses = pd.DataFrame({
    'house_id': [1],
    'latitude': [40.1930597],
    'longitude': [44.5022556]
})

stations = pd.DataFrame({
    'station_id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'station_name': ['Barekamutyun', 'Baghramyan', 'Yeritasardakan', 'Republic Square', 'Zoravar Andranik', 'Sasuntsi Davit', 'Gortsaranain','Shengavit', 'Charbakh', 'Garegin Nzhdeh Square'],
    'latitude': [40.197446,40.1918167,40.1873081, 40.1785331, 40.1699272, 40.1552243, 40.1410611, 40.145224, 40.1405941, 40.1504761],
    'longitude': [44.493467,44.5038091, 44.5191743, 44.5134433, 44.5112603, 44.5058011, 44.4964173, 44.48483, 44.4685333, 44.4816183]
})

# Define a function to calculate the Haversine distance between two points
def haversine(lon1, lat1, lon2, lat2):
    """
    Calculate the great circle distance between two points 
    on the earth (specified in decimal degrees)
    """
    # Convert decimal degrees to radians 
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])

    # Haversine formula 
    dlon = lon2 - lon1 
    dlat = lat2 - lat1 
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a)) 
    r = 6371 # Radius of earth in kilometers. Use 3956 for miles
    return c * r

# Loop through each house and station to find the shortest distance
distances = []
station_ids = []
for house_idx, house in houses.iterrows():
    shortest_distance = float('inf')
    nearest_station_name = ''
    for station_idx, station in stations.iterrows():
        distance = haversine(house['longitude'], house['latitude'], station['longitude'], station['latitude'])
        if distance < shortest_distance:
            shortest_distance = distance
            nearest_station_name = station['station_id']
    distances.append(shortest_distance)
    station_ids.append(nearest_station_name)

# Add the new columns to the houses table
houses['shortest_distance'] = distances
houses['nearest_station_id'] = station_ids

,station_id,station_name,latitude,longitude
0,1,Barekamutyun,40.197446,44.493467
1,2,Baghramyan,40.191817,44.503809
2,3,Yeritasardakan,40.187308,44.519174
3,4,Republic Square,40.178533,44.513443
4,5,Zoravar Andranik,40.169927,44.511260
5,6,Sasuntsi Davit,40.155224,44.505801
6,7,Gortsaranain,40.141061,44.496417
7,8,Shengavit,40.145224,44.484830
8,9,Charbakh,40.140594,44.468533
9,10,Garegin Nzhdeh Square,40.150476,44.481618


In [89]:
for i, house in houses.iterrows():
    distances = []
    print(stations['station_name'][house['nearest_station_id']-1])
    distance = get_walking_distance(house['longitude'], house['latitude'], stations['longitude'][house['nearest_station_id']-1], stations['latitude'][house['nearest_station_id']-1])
    print(distance)
    distances.append(distance)
houses["walk_distance"] = distances
houses

Baghramyan
329


,house_id,latitude,longitude,shortest_distance,nearest_station_id,walk_distance
0,1,40.19306,44.502256,0.19109,2,329


# OpenSreetMap bus stop info

In [97]:
import requests

# Overpass API endpoint to get bus stop locations in Yerevan
url = "http://overpass-api.de/api/interpreter"

# Overpass QL query to retrieve all bus stops in Yerevan
query = """
[out:json];
area["name"="Yerevan"]->.a;
node(area.a)["highway"="bus_stop"];
out;
"""

# send HTTP request to Overpass API
response = requests.post(url, data=query)

# parse JSON response to get bus stop locations
json_response = response.json()
elements = json_response['elements']
print(elements)

bus_stops = []

for element in elements:
    # get bus stop name and coordinates
    name = element['tags']['name']
    lat = element['lat']
    lon = element['lon']

    # add bus stop to list
    bus_stop = {"name": name, "lat": lat, "lon": lon}
    bus_stops.append(bus_stop)

[]


In [94]:
bus_stops

[]